#### Importing Dependencies.
Make sure to create an env using <b> python 3.10 </b> to prevent facing any library issue.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
from tqdm import tqdm
import hazm

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

from torch.utils.data import DataLoader, Dataset 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"This device supports '{device}'")

_______________________________
#### Loading Embedding Model
* Link: https://huggingface.co/heydariAI/persian-embeddings

In [ ]:
embedding_model_path = os.path.join("C:/Users/Amir/Desktop/Models/Embeddings/Persian Embedding")

# Loading model as a high-level helper
# from transformers import pipeline
# pipe = pipeline("feature-extraction", model="heydariAI/persian-embeddings", cache_dir=embedding_model_path)

# Loading model directly
from transformers import AutoTokenizer, AutoModel
embedding_tokenizer = AutoTokenizer.from_pretrained("heydariAI/persian-embeddings", cache_dir=embedding_model_path)
embedding_model = AutoModel.from_pretrained("heydariAI/persian-embeddings", cache_dir=embedding_model_path)
embedding_model.eval()

print("Embedding Model has been loaded successfully.")

_______________________________________
#### Taking Input and Test the Database
* Taking Input as a test and searching through qdrant database for <b> Retrieval </b>

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def text_to_embedding(text, embedding_tokenizer, embedding_model):
    encoded_input = embedding_tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.inference_mode():
        model_output = embedding_model(**encoded_input)
    
    embedding = mean_pooling(model_output, encoded_input["attention_mask"])
    return(embedding.cpu().numpy())

In [ ]:
input_text = "مولکول های اکسیژن به چه صورت عمل میکنند؟"
embedded_input = text_to_embedding(input_text, embedding_tokenizer, embedding_model)

________________________________
#### Loading Qdrant and Search

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

results = client.query_points(
    collection_name="wiki_farsi",
    query=embedded_input.squeeze().tolist(),
    limit=5
)

contexts = [p.payload["text"] for p in results.points]

print(contexts[0])